# Multiple Subject Simulation

The issue with single-subject simulation is that their empirical relative value difference combinations by themselves are not meaningful. Only by collecting data from many subjects with varying parameters is the aDDM able to capture meaningful data, as evidence by Eum et al.'s approach (2023), and my own understanding of the pipeline.

Therefore, this notebook attempts to randomly sample the subjects and produce model-free analysis results consistent with published results. It should be that:

$$
\lim_{n \rightarrow S} \text{pooled model-free analysis} \approx \text{empirical model-free analysis}, \qquad S = \text{total number of subjects},\; n = \text{number of subjects in sample}
$$

I can also see this notebook going towards a direction of sampling procedure where sampling complementary pairs would be descriptive. In this case, we'd have to find a way to define complements in the aDDM space.

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath(".."))

In [ ]:
from ast import literal_eval
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_raw = pd.read_csv('../1ms_trial_data.csv')
df_raw['RT'] = df_raw['RT']*1000 # adjustment for RT
df_raw['fixation'] = df_raw['fixation'].apply(literal_eval)

to_drop = pd.read_csv("../dropped_trials.csv").rename(columns={"parcode": "sub_id"})

df = df_raw[
    ~df_raw.set_index(["sub_id", "trial"]).index.isin(
        to_drop.set_index(["sub_id", "trial"]).index
    )
]

Let's look at model-free analysis using samples of size 15, 30, and 50.

## Pre-simulation model-free analysis

In [ ]:
from pyddm.preprocessing.dataset import rasterize_data
from mfa import plot_basic_psychometrics, plot_fixation_properties

sub_ids = np.unique(df['sub_id'])

for size in ([15, 30, 50]):
    size = 15
    sample_sub_ids = np.random.choice(sub_ids, size=size, replace=False)

    sample_df = df[df["sub_id"].isin(sample_sub_ids)]
    sample_df["choice"] = sample_df["choice"].map({"left": 0, "right": 1})

    samples_rasterized = rasterize_data(sample_df, subject_col='sub_id',trial_col='trial',seq_col='fixation')
    samples_rasterized['fix_dur'] = samples_rasterized.apply(
        lambda r: r['fix_end'] - r['fix_start'],
        axis=1
    )

    samples_rasterized['fix_num'] = (
        samples_rasterized
        .groupby(['sub_id', 'trial'])
        .cumcount() + 1
    )

    samples_rasterized['fix_num_rev'] = (
        samples_rasterized
        .groupby(['sub_id', 'trial'])
        .cumcount(ascending=False) + 1
    )

    print(f'Plotting model-free analysis of randomly sampled subjects size: {size}:\n')
    plot_fixation_properties(samples_rasterized)
    plot_basic_psychometrics(samples_rasterized)

In [ ]:
from simulation import get_corrected_empirical_distributions

size = 30
sample_sub_ids = np.random.choice(sub_ids, size=size, replace=False)

sample_df = df[df["sub_id"].isin(sample_sub_ids)]
sample_df["choice"] = sample_df["choice"].map({"left": 0, "right": 1})

value_diffs = np.arange(-4, 4.25, 0.25)
legend = {
    "left": {1},
    "right": {2},
    "transition": {0}, 
    "blank_fixation": {4}
}
fixation_col = 'fixation'
left_value_col = 'avgWTP_left'
right_value_col = 'avgWTP_right'
cutoff = 0.95

empirical_distributions = get_corrected_empirical_distributions(
    sample_df,
    value_diffs=value_diffs,
    legend=legend,
    fixation_col=fixation_col,
    left_value_col=left_value_col,
    right_value_col=right_value_col,
    cutoff=cutoff
)

## Simulation

In [ ]:
from simulation import generate_fixations

dt = 0.01

trials = sample_df.loc[sample_df['trial'] % 2 == 1].loc[:, ['avgWTP_left', 'avgWTP_right']]
trials['fixation'] = None

rows = []
for idx, r in trials.iterrows():
    fx = generate_fixations(dt, r.avgWTP_left - r.avgWTP_right, empirical_distributions)
    if fx is not None:
        rows.append((idx, fx))

trials_clean = trials.loc[[i for i, _ in rows]].copy()
trials_clean['fixation'] = [fx for _, fx in rows]
my_trials = trials_clean.to_dict(orient="records")

In [ ]:
from simulation import simulate

seed = 42
model_conditions = {'drift_rate': 0.8, 'theta': 0.38, 'noise': 0.63}

results_df = simulate(dt, model_conditions, my_trials, seed=seed, save_results=False)

results_df['sub_id'] = f'seed{seed}_subjects{size}_sim'
results_df['trial'] = range(1, len(trials_clean) + 1)
results_df = results_df.rename(columns={'fixation': 'fix_sequence'})
results_df = results_df.drop(columns = ['trajectory'])
print(np.average(results_df.loc[:, "RT"]))
results_df.head()

## Post-simulation model-free analysis

In [ ]:
simulations_rasterized = rasterize_data(results_df, subject_col='sub_id',trial_col='trial',seq_col='fix_sequence')
simulations_rasterized['fix_dur'] = simulations_rasterized.apply(
    lambda r: r['fix_end'] - r['fix_start'],
    axis=1
)

simulations_rasterized['fix_num'] = (
    simulations_rasterized
    .groupby(['sub_id', 'trial'])
    .cumcount() + 1
)

simulations_rasterized['fix_num_rev'] = (
    simulations_rasterized
    .groupby(['sub_id', 'trial'])
    .cumcount(ascending=False) + 1
)
simulations_rasterized.head()

In [ ]:
plot_basic_psychometrics(simulations_rasterized)
plot_fixation_properties(simulations_rasterized)

## Computing loss on model-free analysis

In [ ]:
from mfa import compute_mfa, model_free_loss

emp_mfa = compute_mfa(samples_rasterized)
sim_mfa = compute_mfa(simulations_rasterized)
loss = model_free_loss(emp_mfa, sim_mfa)
loss

Improvements to quantifying mfa loss can be made by:
- Normalizing each summary to unit empirical variance
- Inspecting per-metric losses (diagnostics)
- Adding profile-likelihood plots per θ
- Testing identifiability by holding one stat out